# Round 4 · The invariant ⚔️ tier D

Three rounds in, one campaign has been invisible to everything you built, and it was never going to show up in an anomaly model. **Pre-hijack:** the attacker registers with the victim's email address first, never verifies it, logs in once or twice and waits. Weeks later the real person finds the account "already exists", resets the password from their own device, verifies the email, adds a card. The attacker's remember-me session survives all of it, and days later it resumes and orders with the victim's new card.

Nothing about any single event is unusual. What is wrong is a **sequence**, and a sequence that should be impossible is an *invariant*: a rule with no threshold and no baseline. The scorer judges this round on what it **adds** to Round 3, not on the cumulative total (`--prev`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../tools"))
from atolab import *
import matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

df = dedupe(load_events())
prev = pd.read_csv("../suspects_r3.csv")      # or the reference: python tools/checkpoint.py --round 3 --catch-up

## 4.1 The textbook invariant, and why it is the wrong one

"A password reset must end every other session." It is the rule every security guide gives, and the first thing to do with a rule is test it on the log you have. Count the accounts that complete a user-initiated reset, and how many of them have a `sessions_revoked_all` anywhere after it.

In [ ]:
resets = df[(df.event_type == "password_reset_completed") & (df.actor == "user")].sort_values("ts").drop_duplicates("account_id")[["account_id", "ts"]]
revokes = df[df.event_type == "sessions_revoked_all"][["account_id", "ts"]].rename(columns={"ts": "revoke_ts"})
m = resets.merge(revokes, on="account_id", how="left")
# TODO: how many reset accounts have NO revoke after the reset? What would the textbook rule flag?
print(len(resets), "accounts completed a user-initiated reset;", m[m.revoke_ts > m.ts].account_id.nunique(), "have a revoke after it")

**Read the result before you write a rule.** If almost no legitimate reset in this store is followed by a revoke, the textbook rule flags almost every owner who ever reset a password. That is a finding in itself (the product does not log, or does not do, the thing the invariant assumes), and it means you must build the invariant from what the log *does* record.

## 4.2 A usable invariant: the owner's email was never verified

Look at the accounts that were created **inside the window** (their `account_created` event is in the log) and later completed a reset. A legitimate sign-up verifies its email early, because the verification link is how the account exists. For the pre-hijack accounts the mail went to the victim, so the verification only arrives *after* the reset the victim runs.

Build three facts per window-born account: when and from which device it was created, when its email was first verified, and its reset. Then write the rule.

In [ ]:
created = df[df.event_type == "account_created"].sort_values("ts").drop_duplicates("account_id")[["account_id", "ts", "device_id"]].rename(columns={"ts": "created_ts", "device_id": "created_device"})
verified = df[df.event_type == "email_verified"].groupby("account_id", observed=True).ts.min().rename("verified_ts")
resets = df[(df.event_type == "password_reset_completed") & (df.actor == "user")][["account_id", "ts", "device_id", "event_id"]]
m = resets.merge(created, on="account_id").merge(verified, on="account_id", how="left")
print(m.account_id.nunique(), "window-born accounts completed a user reset")
# TODO (the invariant): keep the resets with no email_verified before them, from a device other than the one that created the account
p1 = m.iloc[0:0]
print(p1.account_id.nunique(), "accounts violate the invariant")

Note what is missing: **no `n_prior` clause.** Round 1 taught you to require history before believing novelty. Tier D accounts are new by construction, so a known-history clause would delete the whole tier. The same discipline, applied blindly, is the bug.

## 4.3 The symptom is not a rule

The missing revoke has a visible symptom: after the reset, a `session_resume` appears on a device the account used *before* the reset. It looks like the cleanest possible test of "the old session survived". Try it on its own, across **all** accounts, and score it.

In [ ]:
# TODO: reset, then a later session_resume login on a device the account used before the reset
p2 = []
print(len(p2), "accounts")
save_suspects(p2, "../suspects_p2_alone.csv")
score_now("../suspects_p2_alone.csv")

Every owner whose browser rolls a remember-me cookie after a reset looks like a survivor. Read the cohort table: the symptom is the *cause* of nothing, and flagging it alone puts a dozen cohorts far over budget. Keep P2 as a way to *rank* inside P1, never as a rule of its own.

## 4.4 Submit (cumulative) and score the delta

The first event you claim is when your detector could first have pointed at the account. For D that is not the attacker's first login (you could not know it was the attacker's yet); the earliest honest claim is the surviving session that resumes after the reset. Expect poor time-to-detect: the invariant cannot fire until the owner shows up, days or weeks after the compromise.

In [ ]:
first_resume = None   # TODO (optional): the first session_resume on the creating device after the reset, per flagged account
new = pd.DataFrame({"account_id": p1.account_id.unique(), "score": 1.0, "first_event_id": first_resume})
out = pd.concat([new, prev[["account_id", "score", "first_event_id"]]]).drop_duplicates("account_id")
save_suspects(out[["account_id"]], "../suspects_r4.csv", scores=out.score, first_event=out.first_event_id)
score_now("../suspects_r4.csv", "--prev", "../suspects_r3.csv")

## 4.5 Before Round 5

- How many accounts did this round add, and how many were takeovers? (Look at the "This round added" line, not the cumulative precision.)
- The model queue in Round 3 contained some of these accounts by accident. What does that say about relying on a score you do not understand?
- If you could change **one thing in the product** so this campaign cannot happen, what is it? (The deploy slide answers it.)

Fell behind? `python tools/checkpoint.py --round 4 --catch-up` writes the reference `suspects_r4.csv`.